# Step1 导入相关模块

In [ ]:
# md 文档编辑好后esc - m - shift enter

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM,DataCollatorForSeq2Seq, TrainingArguments,Trainer

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import datasets
datasets.__version__

'4.8.5'

In [3]:
import transformers
transformers.__version__


'5.3.0'

In [5]:
import warnings
warnings.filterwarnings('ignore')

# Step2 加载数据集

In [ ]:
ds = load_dataset('json', data_dir="alpaca_data_zh/")
ds = ds['train'] # 加载数据集后放到train数据集中
ds

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 48818
})

In [11]:
# 使用切面查看部分样本数据
ds[:2]

{'instruction': ['保持健康的三个提示。', '三原色是什么？'],
 'input': ['', ''],
 'output': ['以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
  '三原色通常指的是红色、绿色和蓝色（RGB）。它们是通过加色混合原理创建色彩的三种基础颜色。在以发光为基础的显示设备中（如电视、计算机显示器、智能手机和平板电脑显示屏）, 三原色可混合产生大量色彩。其中红色和绿色可以混合生成黄色，红色和蓝色可以混合生成品红色，蓝色和绿色可以混合生成青色。当红色、绿色和蓝色按相等比例混合时，可以产生白色或灰色。\n\n此外，在印刷和绘画中，三原色指的是以颜料为基础的红、黄和蓝颜色（RYB）。这三种颜色用以通过减色混合原理来创建色彩。不过，三原色的具体定义并不唯一，不同的颜色系统可能会采用不同的三原色。']}

# Step3 数据预处理

In [ ]:
# 分词器,langbot
# AutoTokenizer 读取指定模型中文件
# 需要梯子后才能访问下载huggle face
tokenizer = AutoTokenizer.from_pretrained('Langboat/bloom-1b4-zh')
tokenizer

'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/Langboat/bloom-1b4-zh/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/Langboat/bloom-1b4-zh/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/Langboat/bloom-1b4-zh/resolve/main/config.json
Retrying in 4s [Retry 3/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/Langboat/bloom-1b4-zh/resolve/main/config.json
Retrying in 8s [Retry 4/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/Langboat/bloom-1b4-zh/resolve/main/config.json
Retrying in 8s [Retry 5/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https:

OSError: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.

In [15]:
# 使用分词器读取内容并处理样本
def process_func(example):
    # 单条样本最大长度
    MAX_LENGTH = 256
    # 将文本数据交给模型后转为模型能看得懂的数据，模型关注id;transformers相关的大模型都要计算【注意力机制】，所以有attention_mask；
    input_ids, attention_mask, labels = [], [], []
    # 使用tokenizer做分词, 告诉模型你作为Assistant开始给我回答
    instruction = tokenizer("\n".join(["Human:"+example['instruction'], example['input']]).strip() + "\n\nAssistant:")
    # eos_token为结束符，表示这句话说完了
    response = tokenizer(example['output'] + tokenizer.eos_token)

    input_ids = instruction['input_ids'] + response['input_ids']
    attention_mask = instruction['attention_mask'] + response['attention_mask']
    # 只关心后半段 response['input_ids']
    labels = [-100] * len(instruction['input_ids']) + response['input_ids']

    if len(input_ids)> MAX_LENGTH:
        # 大于长度后，对数据做切片操作
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return{
        'input_ids':input_ids, 'attention_mask':attention_mask, 'labels':labels
    }
    

In [ ]:
# 调用函数处理每条样本数据
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

In [ ]:
# 截取部分数据查看
tokenized_ds[:2]

# 解码
tokenizer.decode(tokenized_ds[:2]['input_ids'])

In [17]:
# 解码 labels中不等于 -100 的内容
tokenizer.decode(list[filter(lambda x: x!=-100, tokenized_ds[2]['labels'])])

NameError: name 'tokenizer' is not defined

# Step4 模型创建

In [18]:
# 导入模型
# 
model = AutoModelForCausalLM.from_pretrained("Langboat/bloom-1b4-zh", low_cpu_mem_usage = True)
model

'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/Langbog/bloom-1b4-zh/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/Langbog/bloom-1b4-zh/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/Langbog/bloom-1b4-zh/resolve/main/config.json
Retrying in 4s [Retry 3/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/Langbog/bloom-1b4-zh/resolve/main/config.json
Retrying in 8s [Retry 4/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/Langbog/bloom-1b4-zh/resolve/main/config.json
Retrying in 8s [Retry 5/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://hug

OSError: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.

In [ ]:
# 汇总参数量
sum(param.numel() for param in model.parameters())

# 根据参数量计算需要的显存

# float32 用4个字节存储
model size: 1.36 * 4 ≈ 5.26
# 做全量参数调参
SGD gradient ： 1.36 * 4 ≈ 5.26
with Momentum Optimizer : 1.36 * 4 ≈ 5.26
with Adam Optimizer: 还有计算二阶动量 : 1.36 * 4 ≈ 5.26

# 总共
1.36 * 4 * 4 ≈ 20.8G

# BitFit

In [ ]:
num_param = 0
for name, param in model.named_parameters():
    if 'bias' not in name:
        # name中不包含偏置项， 冻结参数
        param.requires_grad = False
    else:
        # 默认为true，可不用显示写出来
        param.requires_grad = True
        num_param += param.numel()

# 最终需要调整的参数
num_param

# 需要调整的参数占原参数的占比
num_param / sum(param.numel() for param in model.parameters())

NameError: name 'model' is not defined

# Step5 配置训练参数

In [20]:
args = TrainingArguments(
    output_dir='./chatbot', # 输出文件夹，存储模型预测结果和模型文件checkpoints
    per_device_train_batch_size=1, # 训练模型时，每个GPU核/CPU 上面对应的batchSize大小（样本数）
    gradient_accumulation_steps=8, # 执行反向传播/更新参数前，对应梯度计算累计了 n 次
    logging_steps=10, # 每隔 n 次迭代，落地一次日志
    num_train_epochs=1 # 模型学习数据集学习 n 遍
)


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

# Step6 创建训练器

In [21]:
# 获取训练器
trainer = Trainer(
    model= model, # 虽然传的对象，但部分参数已经被冻结了
    args= args,
    train_dataset= tokenized_ds,
    data_collator= DataCollatorForSeq2Seq(tokenizer=tokenizer, padding = True), # 构建一批次数据
)

NameError: name 'model' is not defined

In [ ]:
# 开始训练
trainer.train()

NameError: name 'trainer' is not defined